# Accessing Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

# Demand per capita

## Trial 1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib  # for matplotlib.colormaps
from dateutil.easter import easter

def plot_holiday_per_capita(demand, info, holiday_name, holiday_fn, years, cmap_name='plasma',
                            pop_col='Persons', name_col='Name'):
    """
    Plot per-capita electricity demand for all substations on a chosen holiday across multiple years.

    Parameters
    ----------
    demand : pd.DataFrame
        Half-hourly demand data (index = datetime, columns = substations).
    info : pd.DataFrame
        Metadata for substations, must include population and full name columns.
    holiday_name : str
        Name of the holiday (for titles/labels).
    holiday_fn : function(year) -> pd.Timestamp
        Function that returns the holiday date for a given year.
        Example: lambda y: pd.Timestamp(f"{y}-12-25") for Christmas.
    years : list of int
        Years to include in the plot.
    cmap_name : str
        Matplotlib colormap name (default 'plasma').
    pop_col : str
        Column in info containing population values.
    name_col : str
        Column in info containing full substation names.
    """

    # Ensure datetime index
    demand.index = pd.to_datetime(demand.index)

    substations = demand.columns.tolist()
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    norm = mcolors.Normalize(vmin=min(years), vmax=max(years))

    for substation in substations:
        plt.figure(figsize=(10, 4))
        # Get population and full name from info
        pop = info.loc[substation, pop_col]
        full_name = info.loc[substation, name_col]

        for year in years:
            holiday_date = holiday_fn(year)
            day_data = demand[demand.index.date == holiday_date.date()]
            if day_data.empty:
                continue

            # Fractional hours
            x = day_data.index.hour + day_data.index.minute / 60.0
            # Normalize demand by population
            y = day_data[substation].values / pop

            color = cmap(norm(year))
            plt.plot(x, y, color=color, label=str(year))

        plt.title(f'Per Capita Demand on {holiday_name} – {full_name}')
        plt.xlabel('Time of Day (hour)')
        plt.ylabel('Electricity Demand per Person (kW/person)')
        plt.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize='small')
        plt.xlim(0, 24)
        xticks = np.arange(0, 25, 2)
        plt.xticks(xticks, [f'{t:02d}:00' for t in xticks], rotation=45)
        plt.tight_layout()
        #plt.show()

In [ ]:
plot_holiday_per_capita(
    demand, info,
    holiday_name="Christmas Day",
    holiday_fn=lambda y: pd.Timestamp(f"{y}-12-25"),
    years=list(range(2010, 2017))
)

## Trial 2 (fixing y axis)
- use this one, it saves the plots to folders

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib  # for matplotlib.colormaps
from dateutil.easter import easter
from datetime import timedelta
from pathlib import Path

# -------------------------------
# Plotting function
# -------------------------------

def plot_holiday_per_capita(
    demand, info, holiday_name, holiday_fn, years,
    cmap_name='plasma', pop_col='Persons', name_col='Name',
    unit='W',
    base_dir="/home/565/pv3484/aus_substation_electricity/figures/Demand_capita"
):
    """
    Plot per-capita electricity demand for all substations on a chosen holiday across multiple years.
    Saves figures into holiday-specific folders under base_dir.
    """

    # Ensure datetime index
    demand.index = pd.to_datetime(demand.index)

    substations = demand.columns.tolist()
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    norm = mcolors.Normalize(vmin=min(years), vmax=max(years))

    # Use pathlib for robust path handling
    base_path = Path(base_dir)
    holiday_folder = base_path / holiday_name.replace(" ", "_")
    holiday_folder.mkdir(parents=True, exist_ok=True)

    for substation in substations:
        plt.figure(figsize=(10, 4))
        if substation not in info.index:
            continue

        pop = info.loc[substation, pop_col]
        full_name = info.loc[substation, name_col]
        residential_fraction = info.loc[substation, "Residential"]

        for year in sorted(years):  # keep legend chronological
            holiday_date = holiday_fn(year)
            # safer date matching
            day_data = demand.loc[demand.index.normalize() == pd.Timestamp(holiday_date).normalize()]
            if day_data.empty:
                continue

            # Fractional hours
            x = day_data.index.hour + day_data.index.minute / 60.0

            # Normalise demand by population with chosen unit
            if unit == 'raw':
                y = day_data[substation].values / pop
                ylabel = "Electricity Demand per Person (raw units)"
            elif unit == 'W':
                y = (day_data[substation].values * 1_000_000) / pop
                ylabel = "Electricity Demand per Person (W/person)"
            elif unit == 'kWh':
                # derive interval length dynamically
                interval_hours = (demand.index[1] - demand.index[0]).seconds / 3600
                y = (day_data[substation].values * interval_hours * 1000) / pop
                ylabel = f"Electricity Consumption per Person (kWh per {int(interval_hours*60)} min)"
            else:
                raise ValueError("unit must be 'raw', 'W', or 'kWh'")

            color = cmap(norm(year))
            plt.plot(x, y, color=color, label=str(year))

        # Add residential fraction to title
        plt.title(
            f"{holiday_name} Demand per Capita – {full_name} "
            f"(Residential fraction: {residential_fraction:.2f})"
        )
        plt.xlabel('Time of Day (hour)')
        plt.ylabel(ylabel)
        plt.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize='small')
        plt.xlim(0, 24)
        xticks = np.arange(0, 25, 2)
        plt.xticks(xticks, [f'{t:02d}:00' for t in xticks], rotation=45)
        plt.grid(alpha=0.3)
        plt.tight_layout()

        # Save figure into holiday folder
        filename = f"{holiday_name.replace(' ', '_')}_{substation}.png"
        filepath = holiday_folder / filename
        plt.savefig(filepath, dpi=300)
        print(f"Saved plot to: {filepath}")
        plt.close()

# -------------------------------
# Holiday functions and registry
# -------------------------------

def new_years_day(year): return pd.Timestamp(f"{year}-01-01")
def australia_day(year): return pd.Timestamp(f"{year}-01-26")
def anzac_day(year): return pd.Timestamp(f"{year}-04-25")
def good_friday(year): return easter(year) - timedelta(days=2)
def easter_saturday(year): return easter(year) - timedelta(days=1)
def easter_sunday(year): return easter(year)
def easter_monday(year): return easter(year) + timedelta(days=1)
def christmas_day(year): return pd.Timestamp(f"{year}-12-25")
def boxing_day(year): return pd.Timestamp(f"{year}-12-26")

HOLIDAYS = {
    "New Year’s Day": new_years_day,
    "Australia Day": australia_day,
    "ANZAC Day": anzac_day,
    "Good Friday": good_friday,
    "Easter Saturday": easter_saturday,
    "Easter Sunday": easter_sunday,
    "Easter Monday": easter_monday,
    "Christmas Day": christmas_day,
    "Boxing Day": boxing_day,
}

# -------------------------------
# Loop through all holidays
# -------------------------------

def run_all_holidays(demand, info, years, unit='W', base_dir="/home/565/pv3484/aus_substation_electricity/figures/Demand_capita"):
    for holiday_name, holiday_fn in HOLIDAYS.items():
        plot_holiday_per_capita(
            demand, info,
            holiday_name, holiday_fn,
            years, unit=unit,
            base_dir=base_dir
        )

In [ ]:
# Loop through all holidays defined in HOLIDAYS
for holiday_name, holiday_fn in HOLIDAYS.items():
    plot_holiday_per_capita(
        demand,
        info,
        holiday_name=holiday_name,
        holiday_fn=holiday_fn,
        years=list(range(2004, 2018)),  # adjust year range as needed, if year is changed it will override old plot with new time range
        unit='W',
        base_dir="/home/565/pv3484/aus_substation_electricity/figures/Demand_capita"
    )

## Using function to plot just blakehurst

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import matplotlib
from pathlib import Path

# Save all BLAKE plots into one PDF
pdf_path = Path("/home/565/pv3484/aus_substation_electricity/figures/Demand_capita") / "BLAKE_holiday_plots.pdf"
pdf_path.parent.mkdir(parents=True, exist_ok=True)

# Set up colormap for years
years = range(2004, 2018)
cmap = matplotlib.colormaps.get_cmap("viridis")
norm = plt.Normalize(vmin=min(years), vmax=max(years))

with PdfPages(pdf_path) as pdf:
    for holiday_name, holiday_fn in HOLIDAYS.items():
        plt.figure(figsize=(10, 4))

        if "BLAKE" not in info.index:
            continue

        pop = info.loc["BLAKE", "Persons"]
        full_name = info.loc["BLAKE", "Name"]
        residential_fraction = info.loc["BLAKE", "Residential"]

        for year in years:
            holiday_date = holiday_fn(year)
            day_data = demand.loc[demand.index.normalize() == pd.Timestamp(holiday_date).normalize()]
            if day_data.empty:
                continue

            x = day_data.index.hour + day_data.index.minute / 60.0
            y = (day_data["BLAKE"].values * 1_000_000) / pop
            ylabel = "Electricity Demand per Person (W/person)"

            # Color‑code each year
            color = cmap(norm(year))
            plt.plot(x, y, color=color, label=str(year))

        plt.title(
            f"{holiday_name} Demand per Capita – {full_name} "
            f"(Residential fraction: {residential_fraction:.2f})"
        )
        plt.xlabel("Time of Day (hour)")
        plt.ylabel(ylabel)

        # Legend outside on the right, with color‑coded entries
        plt.legend(
            title="Year",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            fontsize="small"
        )

        plt.xlim(0, 24)
        xticks = np.arange(0, 25, 2)
        plt.xticks(xticks, [f"{t:02d}:00" for t in xticks], rotation=45)
        plt.grid(alpha=0.3)
        plt.tight_layout()

        # Save this figure into the PDF
        pdf.savefig()
        plt.close()

print(f"Saved all BLAKE holiday plots into: {pdf_path}")